# 09 · Business aviation performance analysis

This notebook converts the EUROCONTROL flight records into a business-facing
view of network demand, punctuality, severity and operational recovery.

**Scope**

- Scheduled commercial traffic only: `ICAO Flight Type == 'S'`.
- Directional routes: `ADEP → ADES`.
- Nine monthly snapshots from June 2021 through June 2023.
- March and June 2023 are descriptive here, but remain model holdouts elsewhere.
- Counts refer to operated flights, not passengers, seats or revenue.

The notebook adds a new analysis layer. It does not replace notebooks 01–03.

In [1]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.business_eda import (
    BusinessAnalysisConfig,
    airport_performance,
    business_hypothesis_tests,
    development_flight_paths,
    executive_route_views,
    export_business_analysis,
    hypothesis_test_catalog,
    load_dimension_labels,
    numeric_correlation_table,
    operator_performance,
    overall_kpis,
    plot_departure_arrival_recovery,
    plot_airport_reliability_rankings,
    plot_airport_volume_reliability,
    plot_route_volume_reliability,
    plot_statistical_method_explainer,
    plot_time_reliability_heatmap,
    plot_top_route_comparison,
    read_business_flights,
    route_performance,
    route_threshold_sensitivity,
    scan_route_volume,
)
from src.flight_data_catalog import discover_monthly_flights
from src.flight_data_catalog import (
    compare_null_cohorts,
    profile_nulls_by_month,
    validate_flight_schemas,
)

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
RAW_ICAO = PROJECT_ROOT / "data" / "raw" / "icao"
BASE_OUTPUT_ROOT = PROJECT_ROOT / "reports" / "business_eda"
flight_catalog = discover_monthly_flights(RAW_ROOT)
all_flight_files = [record.path for record in flight_catalog]
assert all_flight_files, "No flight files were found"

## 1. Reproducible business rules

The configuration below makes every reporting choice visible. The executive
route ranking requires at least 500 operated flights and activity in three
observed periods. A broader 100-flight view is retained for network coverage.

Set `RUN_FULL_ANALYSIS = True` when producing the final report. The default
smoke mode reads only a few rows and is safe for rapid validation.

The file catalog combines the new month folders with the legacy flight folder,
prefers one canonical file per month and prevents duplicate ingestion.

In [2]:
CONFIG = BusinessAnalysisConfig(
    test_start="2023-01-01",
    analysis_end_exclusive="2023-07-01",
    executive_min_route_flights=500,
    executive_min_route_periods=3,
    executive_min_operator_flights=1_000,
    route_plot_min_flights=2,
    airport_plot_min_flights=30,
)

RUN_FULL_ANALYSIS = False
SMOKE_ROWS_PER_FILE = 2_000
MAX_ROWS_PER_FILE = None if RUN_FULL_ANALYSIS else SMOKE_ROWS_PER_FILE
OUTPUT_ROOT = (
    BASE_OUTPUT_ROOT
    if RUN_FULL_ANALYSIS
    else PROJECT_ROOT / "reports" / "business_eda_smoke"
)

print({
    "run_full_analysis": RUN_FULL_ANALYSIS,
    "max_rows_per_file": MAX_ROWS_PER_FILE,
    "reporting_end_exclusive": CONFIG.reporting_end,
    "output_root": str(OUTPUT_ROOT),
    "months": [record.month for record in flight_catalog],
})

# Enforce the configured reporting boundary before any CSV reader opens a file.
flight_files = development_flight_paths(all_flight_files, CONFIG)
print({
    "reporting_files": [path.name for path in flight_files],
    "excluded_files": sorted(set(path.name for path in all_flight_files) - set(path.name for path in flight_files)),
})

{'run_full_analysis': False, 'max_rows_per_file': 2000, 'reporting_end_exclusive': '2023-07-01', 'output_root': 'C:\\Users\\celti\\OneDrive - Universidade de Santiago de Compostela\\Verano\\ML_flights_project\\reports\\business_eda_smoke', 'months': ['202106', '202109', '202112', '202203', '202206', '202209', '202212', '202303', '202306']}
{'reporting_files': ['Flights_20210601_20210630.csv.gz', 'Flights_20210901_20210930.csv.gz', 'Flights_20211201_20211231.csv.gz', 'Flights_20220301_20220331.csv.gz', 'Flights_20220601_20220630.csv.gz', 'Flights_20220901_20220930.csv.gz', 'Flights_20230301_20230331.csv.gz', 'Flights_20230601_20230630.csv.gz', 'Flights_20221201_20221231.csv.gz'], 'excluded_files': []}


## 2. Expanded-data contract and null audit

The original source contained six monthly snapshots. The new files add June
and September 2021 plus June 2023. Before calculating business KPIs, the audit
checks that all 18 raw columns still match and compares row-weighted null rates.

A two-percentage-point difference is highlighted for review; it is not an
automatic reason to delete a column. The final full report should rerun this
cell with `RUN_FULL_ANALYSIS = True`.

In [3]:
schema_audit = validate_flight_schemas(all_flight_files)
assert schema_audit["matches_reference_schema"].all(), schema_audit
null_audit = profile_nulls_by_month(
    all_flight_files,
    max_rows_per_file=MAX_ROWS_PER_FILE,
)
reference_months = ["202112", "202203", "202206", "202209", "202212", "202303"]
new_months = ["202106", "202109", "202306"]
null_comparison = compare_null_cohorts(null_audit, reference_months, new_months)

audit_root = OUTPUT_ROOT / "data_quality"
audit_root.mkdir(parents=True, exist_ok=True)
schema_audit.to_csv(audit_root / "schema_compatibility.csv", index=False)
null_audit.to_csv(audit_root / "null_profile_by_month.csv", index=False)
null_comparison.to_csv(audit_root / "null_profile_new_vs_reference.csv", index=False)
display(schema_audit)
display(null_comparison)

,month,file,columns,matches_reference_schema,missing_from_reference,extra_vs_reference
0,202106,Flights_20210601_20210630.csv.gz,18,True,,
1,202109,Flights_20210901_20210930.csv.gz,18,True,,
2,202112,Flights_20211201_20211231.csv.gz,18,True,,
3,202203,Flights_20220301_20220331.csv.gz,18,True,,
4,202206,Flights_20220601_20220630.csv.gz,18,True,,
5,202209,Flights_20220901_20220930.csv.gz,18,True,,
6,202212,Flights_20221201_20221231.csv.gz,18,True,,
7,202303,Flights_20230301_20230331.csv.gz,18,True,,
8,202306,Flights_20230601_20230630.csv.gz,18,True,,


,column,reference_null_pct,new_null_pct,delta_null_pp,material_change
0,AC Registration,0.025,0.116667,0.091667,False
1,AC Operator,0.000,0.000000,0.000000,False
2,AC Type,0.000,0.000000,0.000000,False
3,ACTUAL ARRIVAL TIME,0.000,0.000000,0.000000,False
4,ACTUAL OFF BLOCK TIME,0.000,0.000000,0.000000,False
5,ADEP,0.000,0.000000,0.000000,False
6,ADEP Latitude,0.000,0.000000,0.000000,False
7,ADEP Longitude,0.000,0.000000,0.000000,False
8,ADES,0.000,0.000000,0.000000,False
9,Actual Distance Flown (nm),0.000,0.000000,0.000000,False


## 3. Is a 500-flight route threshold sufficiently inclusive?

This lightweight scan reads only route, period and flight-type columns. It uses
all reporting files even when the rest of the notebook runs in smoke mode.

Two thresholds are useful for different questions:

- **500 flights + 3 periods:** defensible executive comparison.
- **100 flights + 3 periods:** broader network monitoring and discovery.

In [4]:
# Exact volume scan across the configured reporting period.
volume_started = time.perf_counter()
route_volume = scan_route_volume(
    flight_files, CONFIG, max_rows_per_file=MAX_ROWS_PER_FILE
)
volume_sensitivity = route_threshold_sensitivity(route_volume, CONFIG)
display(volume_sensitivity.round(2))
print(f"Route-volume scan: {time.perf_counter() - volume_started:.1f} seconds")

,minimum_flights,minimum_periods,eligible_routes,route_coverage_pct,covered_flights,flight_coverage_pct
0,100,1,0,0.0,0,0.0
1,100,2,0,0.0,0,0.0
2,100,3,0,0.0,0,0.0
3,250,1,0,0.0,0,0.0
4,250,2,0,0.0,0,0.0
5,250,3,0,0.0,0,0.0
6,500,1,0,0.0,0,0.0
7,500,2,0,0.0,0,0.0
8,500,3,0,0.0,0,0.0
9,1000,1,0,0.0,0,0.0


Route-volume scan: 0.2 seconds


### How to interpret the threshold table

`eligible_routes` measures breadth; `flight_coverage_pct` measures how much of
the operated network remains. A high minimum volume increases statistical
stability but removes thin routes. The final report therefore presents both the
executive and broad-coverage views rather than hiding this trade-off.

## 4. Build the compact analytical flight table

The raw compressed files are read in chunks. Only report variables are kept,
continuous values are stored as 32-bit floats, and repeated text fields become
categories. This keeps the full analysis feasible on a low-memory computer.

In [5]:
load_started = time.perf_counter()
flights = read_business_flights(
    flight_files,
    CONFIG,
    chunksize=100_000,
    max_rows_per_file=MAX_ROWS_PER_FILE,
)

# This assertion protects the explicit reporting boundary.
assert flights["FILED OFF BLOCK TIME"].max() < pd.Timestamp(CONFIG.reporting_end)
print({
    "analysis_rows": len(flights),
    "periods": sorted(flights["period"].astype(str).unique()),
    "memory_mb": round(flights.memory_usage(deep=True).sum() / 1024**2, 1),
    "load_seconds": round(time.perf_counter() - load_started, 1),
})

{'analysis_rows': 16089, 'periods': ['2021-06', '2021-09', '2021-12', '2022-03', '2022-06', '2022-09', '2022-12', '2023-03', '2023-06'], 'memory_mb': np.float64(2.6), 'load_seconds': 0.9}


In [6]:
# Compute each table once and reuse it throughout the narrative.
analysis_started = time.perf_counter()
analysis = export_business_analysis(flights, OUTPUT_ROOT, CONFIG)
print({
    "analysis_seconds": round(time.perf_counter() - analysis_started, 1),
    "output_root": str(analysis["output_root"]),
    "outside_reporting_period_rows": 0,
})

{'analysis_seconds': 17.0, 'output_root': 'C:\\Users\\celti\\OneDrive - Universidade de Santiago de Compostela\\Verano\\ML_flights_project\\reports\\business_eda_smoke', 'outside_reporting_period_rows': 0}


## 5. Executive network scorecard

OTP15 is the share of observed arrivals no more than 15 minutes late. Median
delay describes a typical flight; p90 and p95 reveal the operational tail that
drives disruption and customer impact.

In [7]:
kpis = analysis["kpis"]
display(kpis.to_frame("value").round(2))

,value
flights,16089.00
routes,4690.00
operators,230.00
departure_airports,615.00
arrival_airports,534.00
periods,9.00
arrival_observed,16089.00
arrival_otp15_pct,80.71
arrival_delay_median,1.08
arrival_delay_p90,25.57


## 5. Route demand and reliability

Raw percentages are not ranked without a volume rule. Wilson intervals express
uncertainty: small routes receive wider intervals, while high-volume routes are
estimated more precisely.

In [8]:
routes = analysis["routes"]
route_views = analysis["route_views"]

display(routes.head(20).round(2))
display(route_views["popular_reliable"].head(15).round(2))
display(route_views["least_reliable"].head(15).round(2))

,ADEP,ADES,route,flights,periods_active,active_days,arrival_otp15_count,arrival_delay_mean,arrival_delay_median,arrival_delay_p90,...,arrival_delayed_60_count,arrival_otp15_pct,arrival_delayed_15_pct,arrival_delayed_30_pct,arrival_delayed_60_pct,worsened_pct,departed_delayed15,recovered_to_otp15_pct,arrival_otp15_ci_low_pct,arrival_otp15_ci_high_pct
0,KJFK,EGLL,KJFK → EGLL,75,9,9,56,5.140000,3.220000,29.46,...,0,74.67,25.33,9.33,0.00,81.33,9,0.00,63.79,83.14
1,EGNX,EGAA,EGNX → EGAA,42,9,9,41,1.490000,1.870000,10.71,...,0,97.62,2.38,0.00,0.00,54.76,0,NaN,87.68,99.58
2,KLAX,EGLL,KLAX → EGLL,39,9,9,17,18.270000,21.379999,34.54,...,0,43.59,56.41,12.82,0.00,74.36,12,0.00,29.30,59.02
3,OIIE,LTFM,OIIE → LTFM,32,9,9,30,-4.130000,-6.720000,10.86,...,0,93.75,6.25,0.00,0.00,15.62,3,66.67,79.85,98.27
4,KJFK,LFPG,KJFK → LFPG,31,9,9,23,5.120000,-1.320000,39.48,...,1,74.19,25.81,16.13,3.23,58.06,7,0.00,56.75,86.30
5,KEWR,EGLL,KEWR → EGLL,30,7,7,6,38.349998,31.530001,64.24,...,3,20.00,80.00,50.00,10.00,86.67,19,0.00,9.51,37.31
6,EDDK,EGNX,EDDK → EGNX,30,9,9,29,-5.490000,-5.380000,3.13,...,0,96.67,3.33,0.00,0.00,76.67,0,NaN,83.33,99.41
7,RKSI,EDDF,RKSI → EDDF,30,9,9,13,14.110000,16.160000,28.55,...,0,43.33,56.67,10.00,0.00,56.67,11,18.18,27.38,60.80
8,LTAC,LTFJ,LTAC → LTFJ,28,9,9,27,-3.090000,-2.230000,6.66,...,0,96.43,3.57,0.00,0.00,14.29,2,50.00,82.29,99.37
9,KBOS,EGLL,KBOS → EGLL,28,9,9,17,17.790001,9.380000,44.81,...,3,60.71,39.29,21.43,10.71,82.14,8,0.00,42.41,76.43


,ADEP,ADES,route,flights,periods_active,active_days,arrival_otp15_count,arrival_delay_mean,arrival_delay_median,arrival_delay_p90,...,arrival_delayed_60_count,arrival_otp15_pct,arrival_delayed_15_pct,arrival_delayed_30_pct,arrival_delayed_60_pct,worsened_pct,departed_delayed15,recovered_to_otp15_pct,arrival_otp15_ci_low_pct,arrival_otp15_ci_high_pct


,ADEP,ADES,route,flights,periods_active,active_days,arrival_otp15_count,arrival_delay_mean,arrival_delay_median,arrival_delay_p90,...,arrival_delayed_60_count,arrival_otp15_pct,arrival_delayed_15_pct,arrival_delayed_30_pct,arrival_delayed_60_pct,worsened_pct,departed_delayed15,recovered_to_otp15_pct,arrival_otp15_ci_low_pct,arrival_otp15_ci_high_pct


In [9]:
# Volume and reliability answer different business questions, so both are shown.
display(plot_top_route_comparison(routes, CONFIG))
display(plot_route_volume_reliability(routes, float(kpis["arrival_otp15_pct"]), CONFIG))
plt.show()

<Figure size 1400x800 with 2 Axes>

<Figure size 1100x700 with 1 Axes>

C:\Users\celti\AppData\Local\Temp\ipykernel_3332\172737208.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Business interpretation

- High volume + high OTP15: dependable core network.
- High volume + low OTP15: priority for operational intervention.
- Low volume + wide confidence interval: monitor before escalating.
- High p90 with acceptable median: usually reliable but exposed to severe tails.

Every route chart first excludes routes with fewer than two historical flights.
Executive reliability charts then apply the stronger 500-flight / three-period
rule. The two-flight rule prevents singleton routes from appearing as 0% or 100%
reliable while preserving broad volume charts.

The route table remains descriptive. It does not prove that a route causes a
delay because operator, airport, time and duration mix may differ.

## 6. Airline breadth and reliability

`AC Operator` is the operating carrier, not necessarily the marketing airline.
Route count measures network breadth; HHI measures concentration. A high HHI
means that a small number of routes dominate the operator's activity.

In [10]:
operators = analysis["operators"]
eligible_operators = operators.loc[
    operators["flights"].ge(CONFIG.executive_min_operator_flights)
]

display(
    eligible_operators[
        [
            "AC Operator", "flights", "routes", "airports",
            "arrival_otp15_pct", "arrival_delayed_30_pct",
            "arrival_delay_p90", "recovered_to_otp15_pct",
            "route_concentration_hhi",
        ]
    ].head(25).round(2)
)

,AC Operator,flights,routes,airports,arrival_otp15_pct,arrival_delayed_30_pct,arrival_delay_p90,recovered_to_otp15_pct,route_concentration_hhi
0,ZZZ,1751,772,311,88.18,3.43,16.55,28.57,0.0
1,THY,1613,405,217,82.39,6.32,24.66,24.68,0.0


## 7. Which origin and destination airports are most problematic?

Origin and destination roles are analysed separately because they answer
different operational questions. Origin results reflect the conditions under
which a flight begins; destination results reflect the environment into which
the flight arrives. Airports require at least 30 observed flights in these smoke-
safe charts, and every estimate is accompanied by a 95% Wilson interval.

In [11]:
origin_airports = analysis["origin_airports"]
destination_airports = analysis["destination_airports"]

display(origin_airports.head(15).round(2))
display(destination_airports.head(15).round(2))

display(plot_airport_volume_reliability(
    origin_airports, "origin", float(kpis["arrival_otp15_pct"]), CONFIG
))
display(plot_airport_reliability_rankings(origin_airports, "origin", CONFIG))
display(plot_airport_volume_reliability(
    destination_airports, "destination", float(kpis["arrival_otp15_pct"]), CONFIG
))
display(plot_airport_reliability_rankings(destination_airports, "destination", CONFIG))
plt.show()

,role,airport,flights,periods_active,active_days,arrival_otp15_count,arrival_delay_mean,arrival_delay_median,arrival_delay_p90,arrival_delay_p95,...,arrival_delayed_60_count,arrival_otp15_pct,arrival_delayed_15_pct,arrival_delayed_30_pct,arrival_delayed_60_pct,worsened_pct,departed_delayed15,recovered_to_otp15_pct,arrival_otp15_ci_low_pct,arrival_otp15_ci_high_pct
0,origin,EDDP,614,9,9,600,-9.97,-10.48,4.88,9.79,...,0,97.72,2.28,0.00,0.00,36.97,11,45.45,96.21,98.64
1,origin,LTFM,599,9,9,554,-1.16,-2.02,12.82,19.99,...,0,92.49,7.51,1.50,0.00,60.77,34,23.53,90.10,94.34
2,origin,EDDK,492,9,9,469,-6.45,-6.81,8.74,13.99,...,0,95.33,4.67,0.41,0.00,36.18,13,7.69,93.08,96.87
3,origin,LFPG,432,9,9,357,3.88,2.61,20.72,25.91,...,1,82.64,17.36,3.47,0.23,37.50,95,47.37,78.78,85.92
4,origin,KJFK,406,9,9,312,3.48,-1.59,38.87,58.30,...,19,76.85,23.15,14.29,4.68,57.14,85,8.24,72.50,80.69
5,origin,OMDB,332,9,9,115,20.43,18.82,38.35,47.32,...,2,34.64,65.36,18.98,0.60,70.18,166,4.82,29.72,39.91
6,origin,LLBG,264,9,9,227,2.64,0.88,16.60,22.22,...,2,85.98,14.02,1.89,0.76,37.50,31,38.71,81.28,89.66
7,origin,LTFJ,256,9,9,214,5.00,4.76,20.06,24.64,...,0,83.59,16.41,3.12,0.00,66.80,25,4.00,78.57,87.63
8,origin,EGNX,249,9,9,238,-0.07,1.13,11.57,14.05,...,0,95.58,4.42,0.40,0.00,49.40,2,0.00,92.26,97.52
9,origin,EBLG,248,9,9,229,-4.80,-5.21,12.43,18.92,...,0,92.34,7.66,1.61,0.00,27.42,16,25.00,88.35,95.04


,role,airport,flights,periods_active,active_days,arrival_otp15_count,arrival_delay_mean,arrival_delay_median,arrival_delay_p90,arrival_delay_p95,...,arrival_delayed_60_count,arrival_otp15_pct,arrival_delayed_15_pct,arrival_delayed_30_pct,arrival_delayed_60_pct,worsened_pct,departed_delayed15,recovered_to_otp15_pct,arrival_otp15_ci_low_pct,arrival_otp15_ci_high_pct
0,destination,LTFM,768,9,9,573,4.990000,1.82,29.87,40.82,...,14,74.61,25.39,9.90,1.82,19.14,240,27.08,71.41,77.56
1,destination,EGLL,727,9,9,280,25.129999,21.90,57.08,77.41,...,64,38.51,61.49,32.74,8.80,87.35,301,1.66,35.05,42.10
2,destination,EHAM,694,9,9,483,12.510000,7.27,37.93,51.37,...,24,69.60,30.40,14.55,3.46,82.56,144,0.69,66.07,72.90
3,destination,LFPG,686,9,9,457,12.440000,8.17,36.91,49.68,...,22,66.62,33.38,13.27,3.21,67.06,173,9.25,63.01,70.05
4,destination,EDDF,657,9,9,537,1.520000,-1.73,24.09,35.56,...,12,81.74,18.26,6.39,1.83,17.20,157,33.12,78.60,84.50
5,destination,LTFJ,505,9,9,450,-0.990000,-2.58,15.74,25.92,...,3,89.11,10.89,3.56,0.59,27.92,63,42.86,86.09,91.54
6,destination,LEMD,431,9,9,307,8.890000,6.88,33.05,39.22,...,6,71.23,28.77,12.06,1.39,51.51,105,14.29,66.78,75.30
7,destination,EDDM,388,9,9,343,-0.250000,-2.00,17.62,26.61,...,1,88.40,11.60,4.38,0.26,32.22,46,15.22,84.83,91.22
8,destination,LPPT,280,9,9,194,9.500000,7.58,27.99,33.73,...,5,69.29,30.71,8.21,1.79,61.79,63,6.35,63.65,74.40
9,destination,LTAI,246,9,9,194,6.400000,6.32,22.44,29.23,...,1,78.86,21.14,4.47,0.41,57.72,45,20.00,73.34,83.50


<Figure size 1100x700 with 2 Axes>

<Figure size 1400x800 with 2 Axes>

<Figure size 1100x700 with 2 Axes>

<Figure size 1400x800 with 2 Axes>

C:\Users\celti\AppData\Local\Temp\ipykernel_3332\820635734.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Airport-chart interpretation

- The volume–reliability charts show scale, OTP15 and >30-minute exposure.
- The ranking charts use Wilson intervals, not raw percentages.
- A problematic origin is associated with weaker arrival outcomes for flights
  leaving that airport; a problematic destination is associated with weaker
  outcomes for flights arriving there.
- These are unadjusted associations. Route, operator, time and duration mix can
  explain part of the difference and should be controlled before assigning cause.

## 8. When is the network least reliable?

The heatmap uses scheduled departure time, which is known in advance. It helps
identify operational windows for staffing, disruption monitoring and customer
communications. It is descriptive and should not be interpreted as causal.

In [12]:
display(plot_time_reliability_heatmap(flights))
plt.show()

<Figure size 1400x600 with 2 Axes>

C:\Users\celti\AppData\Local\Temp\ipykernel_3332\1303069179.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Delay propagation and recovery

Each hexagon groups many flights. The green diagonal represents equal departure
and arrival delay. Points below it recovered minutes; points above it worsened
after departure.

In [13]:
display(plot_departure_arrival_recovery(flights))
plt.show()

<Figure size 800x700 with 2 Axes>

C:\Users\celti\AppData\Local\Temp\ipykernel_3332\3122058065.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Correlation: association, not causation

Pearson measures linear association. Spearman measures monotonic association and
is less sensitive to extreme delays. Post-event variables are valid for this
retrospective report but remain unavailable to the T−60 prediction model.

In [14]:
correlations = analysis["correlations"]
display(
    correlations.reindex(
        correlations["spearman_rho"].abs().sort_values(ascending=False).index
    ).head(20).round(4)
)

,variable_1,variable_2,rows,pearson_r,pearson_p,spearman_rho,spearman_p
17,schedule_buffer_min,recovery_minutes,16089,1.0000,0.0,1.0000,0.0
0,scheduled_duration_min,actual_duration_min,16089,0.9991,0.0,0.9947,0.0
8,actual_duration_min,Actual Distance Flown (nm),16089,0.9952,0.0,0.9924,0.0
2,scheduled_duration_min,Actual Distance Flown (nm),16089,0.9948,0.0,0.9883,0.0
25,Departure_Delay_Min,Arrival_Delay_Min,16089,0.8938,0.0,0.8297,0.0
27,Arrival_Delay_Min,recovery_minutes,16089,-0.4670,0.0,-0.5253,0.0
16,schedule_buffer_min,Arrival_Delay_Min,16089,-0.4670,0.0,-0.5253,0.0
20,Actual Distance Flown (nm),Arrival_Delay_Min,16089,0.4266,0.0,0.4407,0.0
11,actual_duration_min,Arrival_Delay_Min,16089,0.4249,0.0,0.4323,0.0
5,scheduled_duration_min,Arrival_Delay_Min,16089,0.4081,0.0,0.3933,0.0


## 11. Hypothesis tests: what exactly is being tested?

The catalog below separates tests already automated for the core report from
optional tests that require a business decision before execution.

**Core tests already implemented**

- **H01 — December change:** H0 says December 2021 and December 2022 have equal
  arrival OTP15. A two-proportion z-test is paired with the percentage-point
  difference.
- **H02 — En-route recovery:** H0 says median recovery equals zero among flights
  leaving more than 15 minutes late. Wilcoxon is used because delay differences
  are skewed and contain extreme events.
- **H03 — Haul bands:** H0 says all flight-duration bands share the same arrival-
  delay distribution. Kruskal–Wallis avoids a normality assumption.

**Recommended optional tests for selection**

- H04/H05: origin- and destination-airport association with OTP15.
- H06: operator association with OTP15, with a route-mix warning.
- H07: per-route stability across observed periods, corrected for multiple tests.
- H08: differences across scheduled departure-hour bands.
- H09: monotonic relationship between duration and delay.
- H10: adjusted operator effects after controlling for route, time and duration.

Large datasets can produce tiny p-values for unimportant differences. The report
therefore shows effect size and units alongside significance. Benjamini–Hochberg
is used only when many related hypotheses are tested simultaneously.

In [15]:
h0_catalog = analysis["hypothesis_test_catalog"]
hypothesis_tests = analysis["hypothesis_tests"]
display(h0_catalog)
display(hypothesis_tests.round(5))
display(plot_statistical_method_explainer())
plt.show()

,test_id,business_question,null_hypothesis,method,effect_size,recommendation,implementation
0,H01,Did December punctuality change between 2021 a...,December 2021 and December 2022 have equal arr...,Two-proportion z-test,Difference in percentage points,Core report,Automated
1,H02,Do flights leaving >15 minutes late recover ti...,Median en-route recovery equals zero minutes.,Paired Wilcoxon signed-rank test,Median minutes recovered,Core report,Automated
2,H03,Does the arrival-delay distribution differ by ...,All haul bands have the same arrival-delay dis...,Kruskal-Wallis test,Epsilon squared,Core report,Automated
3,H04,Is origin-airport punctuality associated with ...,Arrival OTP15 is independent of origin airport.,Chi-square test of independence,Cramér's V,Recommended optional,Awaiting selection
4,H05,Is destination-airport punctuality associated ...,Arrival OTP15 is independent of destination ai...,Chi-square test of independence,Cramér's V,Recommended optional,Awaiting selection
5,H06,Is punctuality associated with the operating c...,Arrival OTP15 is independent of AC Operator.,Chi-square test of independence,Cramér's V,Optional; route mix is a confounder,Awaiting selection
6,H07,Are route reliability rates stable across obse...,"For each eligible route, OTP15 is equal across...",Per-route chi-square tests + Benjamini-Hochberg,Maximum percentage-point change,Recommended optional,Awaiting selection
7,H08,Does punctuality differ across scheduled depar...,Arrival-delay distributions are equal across h...,Kruskal-Wallis test,Epsilon squared,Optional,Awaiting selection
8,H09,Is scheduled duration monotonically associated...,Spearman rho between duration and arrival dela...,Spearman rank-correlation test,Spearman rho,Optional; already reported in correlations,Available in correlation table
9,H10,Do operator differences remain after controlli...,Adjusted operator effects on P(delay >15) are ...,Adjusted logistic regression + likelihood-rati...,Adjusted odds ratios,Best for fair operator comparison,Future adjusted analysis


,test,test_id,null_hypothesis,statistic,p_value,effect,effect_unit,p_value_bh
0,December OTP15 equality,H01,December 2021 and December 2022 have equal OTP15,2.806230e+00,0.00501,3.97315,percentage points (2021 minus 2022),0.00698
1,En-route recovery,H02,Median recovery is zero for flights departing ...,1.636134e+06,0.00698,-0.70000,median minutes recovered,0.00698
2,Delay equality across haul bands,H03,Arrival-delay distributions are equal across d...,2.703270e+03,0.00000,0.16788,Kruskal-Wallis epsilon squared,0.00000


<Figure size 1500x480 with 3 Axes>

C:\Users\celti\AppData\Local\Temp\ipykernel_3332\4132739489.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Generate report-ready assets

The export keeps tables, statistical outputs and figures separate. The future
Word report can therefore be regenerated without copying values manually.

In [16]:
print({
    "output_root": str(analysis["output_root"]),
    "tables": sorted(path.name for path in (OUTPUT_ROOT / "tables").glob("*.csv")),
    "figures": sorted(path.name for path in (OUTPUT_ROOT / "figures").glob("*.png")),
    "outside_reporting_period_rows": 0,
})

{'output_root': 'C:\\Users\\celti\\OneDrive - Universidade de Santiago de Compostela\\Verano\\ML_flights_project\\reports\\business_eda_smoke', 'tables': ['destination_airport_performance.csv', 'least_reliable_routes.csv', 'network_kpis.csv', 'operator_performance.csv', 'origin_airport_performance.csv', 'popular_reliable_routes.csv', 'route_performance.csv', 'route_threshold_sensitivity.csv'], 'figures': ['departure_arrival_recovery.png', 'destination_airport_reliability_rankings.png', 'destination_airport_volume_reliability.png', 'origin_airport_reliability_rankings.png', 'origin_airport_volume_reliability.png', 'route_volume_reliability.png', 'statistical_method_explainer.png', 'time_reliability_heatmap.png', 'top_route_comparison.png'], 'outside_reporting_period_rows': 0}


## 13. Reporting limitations

1. The expanded data contains nine separated monthly snapshots, not a continuous calendar.
2. The descriptive report includes March and June 2023. Model notebooks keep them outside fitting and tuning.
3. Results describe operated flights; cancellations and diversions are absent.
4. Flight volume is not passenger, seat or revenue volume.
5. Operator comparisons are affected by route, airport and schedule mix.
6. `STATFOR Market Segment` loses detail in later files and must be interpreted cautiously.
7. Statistical significance does not establish operational causality.

Recommended next data: consecutive months, cancellations, aircraft capacity,
airport constraints, ATFM regulations and weather after the flight-only baseline
is fully documented.